In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
import random

batch_size = 128
epochs = 10
lr = 1e-3
mc_samples = 20
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

In [2]:
class CNN(nn.Module):
    def __init__(self, dropout_p=0.3, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(p=dropout_p)

        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x, return_penult=False):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.dropout(x)

        x = self.pool(F.relu(self.conv2(x)))
        x = self.dropout(x)

        x = self.pool(F.relu(self.conv3(x)))
        x = self.dropout(x)

        x = x.view(x.size(0), -1)
        penult = F.relu(self.fc1(x))
        x = self.dropout(penult)
        logits = self.fc2(x)

        if return_penult:
            return logits, penult
        return logits

    def forward_react(self, x, c):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.dropout(x)

        x = self.pool(F.relu(self.conv2(x)))
        x = self.dropout(x)

        x = self.pool(F.relu(self.conv3(x)))
        x = self.dropout(x)

        x = x.view(x.size(0), -1)
        penult = F.relu(self.fc1(x))
        penult = torch.clamp(penult, max=c)
        x = self.dropout(penult)
        logits = self.fc2(x)
        return logits

In [3]:
transform_cifar = transforms.Compose([
    transforms.ToTensor()
])

transform_mnist = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.expand(3, -1, -1))
])

train_id = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_cifar)
test_id = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_cifar)
test_ood = datasets.MNIST(root='./data', train=False, download=True, transform=transform_mnist)


indices = list(range(len(train_id)))
random.shuffle(indices)
val_size = 5000
val_indices = indices[:val_size]
train_indices = indices[val_size:]

train_subset = Subset(train_id, train_indices)
val_subset = Subset(train_id, val_indices)

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_id_loader = DataLoader(test_id, batch_size=batch_size, shuffle=False)
test_ood_loader = DataLoader(test_ood, batch_size=batch_size, shuffle=False)

100%|██████████| 170M/170M [02:21<00:00, 1.21MB/s] 
100%|██████████| 9.91M/9.91M [00:04<00:00, 2.43MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 212kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.58MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.0MB/s]


In [4]:
def train(model, train_loader, epochs, lr):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_function = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        correct_samples = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()

            logits = model(x)
            loss = loss_function(logits, y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            preds = logits.argmax(dim=1)

            correct_samples += (preds == y).sum().item()

        print(f'Epoch {epoch}; Train loss {total_loss / len(train_loader)}; Accuracy {correct_samples / len(train_loader.dataset) * 100:.2f}%')

In [5]:
def get_react_ood_scores(model, id_loader, ood_loader, c):
    model.to(device)
    model.eval()

    id_scores = []
    ood_scores = []

    with torch.no_grad():
        for x, _ in id_loader:
            x = x.to(device)
            logits = model.forward_react(x, c)

            scores = -torch.logsumexp(logits, dim=1)
            id_scores.append(scores.cpu().numpy())

    with torch.no_grad():
        for x, _ in ood_loader:
            x = x.to(device)
            logits = model.forward_react(x, c)
            scores = -torch.logsumexp(logits, dim=1)
            ood_scores.append(scores.cpu().numpy())

    id_scores = np.concatenate(id_scores)
    ood_scores = np.concatenate(ood_scores)

    return id_scores, ood_scores

In [6]:
def get_softmax_ood_scores(model, id_loader, ood_loader):
    model.to(device)
    model.eval()

    id_scores = []
    ood_scores = []

    with torch.no_grad():
        for x, _ in id_loader:
            x = x.to(device)
            logits = model(x)
            probs = F.softmax(logits, dim=1)
            max_probs, _ = probs.max(dim=1)
            scores = 1.0 - max_probs

            id_scores.append(scores.cpu().numpy())

    with torch.no_grad():
        for x, _ in ood_loader:
            x = x.to(device)
            logits = model(x)
            probs = F.softmax(logits, dim=1)
            max_probs, _ = probs.max(dim=1)
            scores = 1.0 - max_probs

            ood_scores.append(scores.cpu().numpy())

    id_scores = np.concatenate(id_scores)
    ood_scores = np.concatenate(ood_scores)

    return id_scores, ood_scores

def compute_ood_metrics(id_scores, ood_scores):
    y_true = np.concatenate([
        np.zeros_like(id_scores),
        np.ones_like(ood_scores)
    ])
    scores = np.concatenate([id_scores, ood_scores])

    auroc = roc_auc_score(y_true, scores)
    aupr = average_precision_score(y_true, scores)

    fpr, tpr, _ = roc_curve(y_true, scores)
    target_tpr = 0.95
    idxs = np.where(tpr >= target_tpr)[0]
    if len(idxs) > 0:
        fpr95 = fpr[idxs[0]]
    else:
        fpr95 = 1.0

    print(f'AUROC {auroc}')
    print(f'AUPR {aupr}')
    print(f'FPR@95%TPR {fpr95}')

    return auroc, aupr, fpr95

In [7]:
def get_softmax_ood_scores(model, id_loader, ood_loader):
    model.to(device)
    model.eval()

    id_scores = []
    ood_scores = []

    with torch.no_grad():
        for x, _ in id_loader:
            x = x.to(device)
            logits = model(x)
            probs = F.softmax(logits, dim=1)
            max_probs, _ = probs.max(dim=1)
            scores = 1.0 - max_probs

            id_scores.append(scores.cpu().numpy())

    with torch.no_grad():
        for x, _ in ood_loader:
            x = x.to(device)
            logits = model(x)
            probs = F.softmax(logits, dim=1)
            max_probs, _ = probs.max(dim=1)
            scores = 1.0 - max_probs

            ood_scores.append(scores.cpu().numpy())

    id_scores = np.concatenate(id_scores)
    ood_scores = np.concatenate(ood_scores)

    return id_scores, ood_scores


def get_mcd_ood_entropy(model, x, T=20):
    model.to(device)
    model.train()

    with torch.no_grad():
        probs_T = []
        for _ in range(T):
            logits = model(x)
            probs = F.softmax(logits, dim=1)

            probs_T.append(probs.unsqueeze(0))

        probs_T = torch.cat(probs_T, dim=0)

    p_mean = probs_T.mean(dim=0)

    eps = 1e-8
    entropy = -torch.sum(p_mean * torch.log(p_mean + eps), dim=1)

    return entropy


def get_mcd_ood_scores(model, id_loader, ood_loader, T=20):
    model.to(device)

    id_scores = []
    ood_scores = []

    for x, _ in id_loader:
        x = x.to(device)
        entropy = get_mcd_ood_entropy(model, x, T=T)

        id_scores.append(entropy.cpu().numpy())

    for x, _ in ood_loader:
        x = x.to(device)
        entropy = get_mcd_ood_entropy(model, x, T=T)

        ood_scores.append(entropy.cpu().numpy())

    id_scores = np.concatenate(id_scores)
    ood_scores = np.concatenate(ood_scores)

    return id_scores, ood_scores

In [8]:
model = CNN(dropout_p=0.3, num_classes=10)
train(model, train_loader, epochs=epochs, lr=lr)

Epoch 1; Train loss 1.7234641306779601; Accuracy 36.40%
Epoch 2; Train loss 1.375025887042284; Accuracy 49.98%
Epoch 3; Train loss 1.2416355606507172; Accuracy 55.45%
Epoch 4; Train loss 1.1439778032627972; Accuracy 59.08%
Epoch 5; Train loss 1.0797684097831899; Accuracy 61.75%
Epoch 6; Train loss 1.0185115354304963; Accuracy 63.98%
Epoch 7; Train loss 0.973657990721139; Accuracy 65.59%
Epoch 8; Train loss 0.9220463692803275; Accuracy 67.45%
Epoch 9; Train loss 0.8921188823878765; Accuracy 68.38%
Epoch 10; Train loss 0.8673098451373252; Accuracy 69.45%


In [9]:

model.eval()
penults = []
with torch.no_grad():
    for x, _ in val_loader:
        x = x.to(device)
        _, pen = model(x, return_penult=True)
        penults.append(pen.cpu().numpy())

penults = np.concatenate(penults)
all_acts = penults.flatten()
c = np.percentile(all_acts, 90)
print(f'Порог клиппинга c: {c}')

Порог клиппинга c: 0.981294572353363


In [10]:
softmax_id_scores, softmax_ood_scores = get_softmax_ood_scores(model, test_id_loader, test_ood_loader)
auroc, aupr, fpr95 = compute_ood_metrics(softmax_id_scores, softmax_ood_scores)
print(f'Softmax - AUROC: {auroc:.4f}, AUPR: {aupr:.4f}, FPR@95%TPR: {fpr95:.4f}')

AUROC 0.6210456900000001
AUPR 0.5568377495675344
FPR@95%TPR 0.7637
Softmax - AUROC: 0.6210, AUPR: 0.5568, FPR@95%TPR: 0.7637


In [11]:
mcd_id_scores, mcd_ood_scores = get_mcd_ood_scores(model, test_id_loader, test_ood_loader, T=mc_samples)
auroc, aupr, fpr95 = compute_ood_metrics(mcd_id_scores, mcd_ood_scores)
print(f'MCD - AUROC: {auroc:.4f}, AUPR: {aupr:.4f}, FPR@95%TPR: {fpr95:.4f}')

AUROC 0.7044437400000001
AUPR 0.6138299154133684
FPR@95%TPR 0.6363
MCD - AUROC: 0.7044, AUPR: 0.6138, FPR@95%TPR: 0.6363


In [12]:
react_id_scores, react_ood_scores = get_react_ood_scores(model, test_id_loader, test_ood_loader, c)
auroc, aupr, fpr95 = compute_ood_metrics(react_id_scores, react_ood_scores)
print(f'ReAct - AUROC: {auroc:.4f}, AUPR: {aupr:.4f}, FPR@95%TPR: {fpr95:.4f}')

AUROC 0.94709423
AUPR 0.9203564903533042
FPR@95%TPR 0.1513
ReAct - AUROC: 0.9471, AUPR: 0.9204, FPR@95%TPR: 0.1513


In [13]:
metrics = {
    'Method': ['Softmax', 'MCD', 'ReAct'],
    'AUROC': [
        roc_auc_score(np.concatenate([np.zeros_like(softmax_id_scores), np.ones_like(softmax_ood_scores)]), np.concatenate([softmax_id_scores, softmax_ood_scores])),
        roc_auc_score(np.concatenate([np.zeros_like(mcd_id_scores), np.ones_like(mcd_ood_scores)]), np.concatenate([mcd_id_scores, mcd_ood_scores])),
        roc_auc_score(np.concatenate([np.zeros_like(react_id_scores), np.ones_like(react_ood_scores)]), np.concatenate([react_id_scores, react_ood_scores]))
    ],
    'AUPR': [
        average_precision_score(np.concatenate([np.zeros_like(softmax_id_scores), np.ones_like(softmax_ood_scores)]), np.concatenate([softmax_id_scores, softmax_ood_scores])),
        average_precision_score(np.concatenate([np.zeros_like(mcd_id_scores), np.ones_like(mcd_ood_scores)]), np.concatenate([mcd_id_scores, mcd_ood_scores])),
        average_precision_score(np.concatenate([np.zeros_like(react_id_scores), np.ones_like(react_ood_scores)]), np.concatenate([react_id_scores, react_ood_scores]))
    ],
    'FPR@95%TPR': [
        roc_curve(np.concatenate([np.zeros_like(softmax_id_scores), np.ones_like(softmax_ood_scores)]), np.concatenate([softmax_id_scores, softmax_ood_scores]))[0][np.where(roc_curve(np.concatenate([np.zeros_like(softmax_id_scores), np.ones_like(softmax_ood_scores)]), np.concatenate([softmax_id_scores, softmax_ood_scores]))[1] >= 0.95)[0][0]],
        roc_curve(np.concatenate([np.zeros_like(mcd_id_scores), np.ones_like(mcd_ood_scores)]), np.concatenate([mcd_id_scores, mcd_ood_scores]))[0][np.where(roc_curve(np.concatenate([np.zeros_like(mcd_id_scores), np.ones_like(mcd_ood_scores)]), np.concatenate([mcd_id_scores, mcd_ood_scores]))[1] >= 0.95)[0][0]],
        roc_curve(np.concatenate([np.zeros_like(react_id_scores), np.ones_like(react_ood_scores)]), np.concatenate([react_id_scores, react_ood_scores]))[0][np.where(roc_curve(np.concatenate([np.zeros_like(react_id_scores), np.ones_like(react_ood_scores)]), np.concatenate([react_id_scores, react_ood_scores]))[1] >= 0.95)[0][0]]
    ]
}

import pandas as pd


summary_df = pd.DataFrame(metrics)
print("Итоговая таблица метрик:")
print(summary_df)



Итоговая таблица метрик:
    Method     AUROC      AUPR  FPR@95%TPR
0  Softmax  0.621046  0.556838      0.7637
1      MCD  0.704444  0.613830      0.6363
2    ReAct  0.947094  0.920356      0.1513


ReAct показывает явное преимущество по всем метрикам, MCD заметно улучшает качество по сравнению с базовым Softmax, Softmax самый слабый метод